<a href="https://colab.research.google.com/github/hibernator11/adaptation-GLAM-notebooks/blob/main/wacad_gemini_nls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install transformers chromadb

Load modules

First, let's import the required packages for embedding and vector storage.

In [18]:
# Import embedding and vector database libraries
from sentence_transformers import SentenceTransformer
import chromadb
import os
from chromadb.utils import embedding_functions

res_folder = "nls-data/"

# Configuration for vector database
db_collection_name = "nls-data-foundry2"  # Name for our vector collection
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"  # Pre-trained embedding model
path_chroma = os.path.join(res_folder, "chroma_db")  # Path to store the vector database
input_content_dir = os.path.join(res_folder, "nls-sample")  # Directory with cleaned text files

print(path_chroma)

nls-data/chroma_db


Download the dataset and extract

In [ ]:
import requests
import zipfile
from io import BytesIO

# Ensure directory exists
os.makedirs(input_content_dir, exist_ok=True)

# Download the zip file
response = requests.get("https://nlsfoundry.s3.amazonaws.com/text/nls-text-indiaPapers.zip")
response.raise_for_status()  # ensure download worked

# Unzip directly into res_folder
with zipfile.ZipFile(BytesIO(response.content)) as z:
    z.extractall(res_folder)

# Rename the folder
original_unzip_path = os.path.join(res_folder, "nls-text-indiaPapers")
os.rename(original_unzip_path, input_content_dir)

Optionally, you can reduce the number of files to process from the dataset, if it is quite large. This cell keeps a random sample of five files from the dataset, and deletes the rest.

In [ ]:
import random

# Optional - set seed for reproducibility
random.seed(42)

# Get all txt files in the directory
all_files = [
    f for f in os.listdir(input_content_dir)
    if f.endswith(".txt")
]

# Randomly select 5 files to keep
files_to_keep = set(random.sample(all_files, min(5, len(all_files))))

# Delete everything else
for f in all_files:
    if f not in files_to_keep:
        os.remove(os.path.join(input_content_dir, f))

ChromaDB is a vector database designed for efficient storage and retrieval of embeddings. It allows us to perform similarity searches across our corpus.

In [19]:
# Initialize ChromaDB client and collection
client = chromadb.PersistentClient(path=path_chroma)

embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
     model_name=embedding_model
)

collection = client.get_or_create_collection(name=db_collection_name,
                                             embedding_function=embedding_func)

# Load the sentence transformer model for creating embeddings
model = SentenceTransformer(embedding_model, token=False)

# Get all cleaned text files
files = sorted([f for f in os.listdir(input_content_dir) if f.endswith(".txt")])

# Process each file and add its contents to the vector database
for fname in files:
    # Load content file
    content_file_path = os.path.join(input_content_dir, fname)
    with open(content_file_path, encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f]

    # Create embeddings for each line of text
    # This converts text into numerical vectors that capture semantic meaning
    embeddings = model.encode(lines, show_progress_bar=True, convert_to_numpy=True)

    # Add embeddings and metadata to ChromaDB
    #try:
    collection.add(
        ids=[f"{fname}_{i}" for i in range(len(lines))],
        embeddings=embeddings.tolist(),
        documents=lines,
        metadatas=[{"filename": fname} for i in range(len(lines))]
    )
    #except ValueError:
    #    print(f"⚠️ Skipped {fname} due to ValueError")

print("✅ Indexed all lines into Chroma!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Indexed all lines into Chroma!


In [20]:
def semantic_search(query, n_results=2):
    """Perform semantic search on the corpus using vector embeddings.

    Args:
        query: The search query as a string
        n_results: Number of results to return (default: 5)

    Returns:
        Dictionary containing search results from ChromaDB
    """
    # Load the embedding model
    model = SentenceTransformer(embedding_model)

    # Connect to the vector database
    client = chromadb.PersistentClient(path=path_chroma)
    collection = client.get_or_create_collection(name=db_collection_name)

    # Convert query to embedding vector
    query_emb = model.encode([query], convert_to_numpy=True).tolist()

    # Search for similar content in the database
    results = collection.query(
        query_embeddings=query_emb,
        n_results=n_results
    )

    # Display results with metadata
    for text, meta in zip(results["documents"][0], results["metadatas"][0]):
        print(f"{meta['filename']}")
        print(f"→ {text[:]}\n")

    return results

In [21]:
# Example semantic search query
# This demonstrates finding content related to diseases without requiring exact keyword matches
#res = semantic_search("Which hospitals treated leprosy?")
#res = semantic_search("infected villages?", 2)
#res = semantic_search("can you tell me the periods of vaccination?", 5)
#res = semantic_search("major diseases", 5)
res = semantic_search("efforts to mitigate the spread of diseases", 5)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


77505999.txt
→ ?THE MADRAS  PLAGUE  REGULATIONS  IN FORCE OUTSIDE THE PRESIDENCY TOWN. I THE  EPIDEMIC  DISEASES ACT (III OF 1897). An Act to provide for the better prevention of the spread of Dangerous Epidemic Disease. WHEREAS it is expedient to provide for the better prevention of the spread of dangerous epidemic disease; It is hereby enacted as follows:- Short title, extent and commencement. 1. (1) This Act may be called the Epidemic Diseases Act, 1897. (2) It extends to the whole of British India (inclusive of Upper Burma, British Baluchistan, the Santal Parganas and the Pargarna of Spiti); and (3) It shall come into force at once. Power to take special measures and prescribe regulations as to dangerous epidemic disease. 2. (1) When at any time the Governor-General in Council is satisfied that India or any part thereof is visited by, or threatened with, an outbreak of any dangerous epidemic disease, the Governor-General in Council, if he thinks that the ordinary provisions of the 

Using Google Gemini to build an assistant

In [22]:
from google.colab import ai

def generate_answer_gemini(context, question):
    prompt = f"""You are a helpful assistant. Use the following context to answer the question.

Context:
{context}

Question: {question}
Answer:"""

    response = ai.generate_text(prompt, model_name="google/gemini-2.5-flash", stream=False)
    return response

In [23]:
def rag_query_gemini(query, n_results=5):
    results = semantic_search(query, n_results=n_results)
    context = "\n\n".join(results["documents"][0])
    answer = generate_answer_gemini(context, query)

    print("\n=== ANSWER ===")
    print(answer)
    print("\n=== SOURCES ===")
    for meta in results["metadatas"][0]:
        print(f"- {meta['filename']} ")

In [24]:
rag_query_gemini("efforts to mitigate the spread of diseases?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


77505993.txt
→ ?APP. IX.] spread of infection by sea. 319 course of the epidemic. These communications should be made at least once a week. The reports concerning the outbreak and course of the disease should be as complete as possible. They should, in particular, state the measures taken to check the spread of the epidemic and should give, in detail, the preventive measures adopted, with regard to- sanitary or medical inspection; isolation; disinfection; and the measures prescribed with regard to the departure of ships, and the export of susceptible articles. It is to be understood that neighbouring countries reserve to them- selves the right to make special arrangements, with the object of organising an exchange of direct information between the principal administrative officers on their frontiers. The Government of each country must publish, immediately, the measures which it decides to adopt with regard to arrivals from an infected country or local area. The information must be at 

In [25]:
rag_query_gemini("cities in which diseases were spread?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


77505993.txt
→ ?426 Measures to prevent the spread of [APP. X. No. 394, dated the 17th March 1897. From-The Civil Surgeon, Sukkur, To-The Commissioner in Sind. In continuation of my letter No. 341, dated 11th instant, I have the honour to state that for the week ending 15th March 1897 fifteen fresh cases of plague have occurred, making a total of 29 from the beginning of the outbreak, of which 22 have proved fatal and 7 are under treatment at the Contagious Landhies. 2. Of the 15 fresh cases, there is distinct evidence that 13 either had communication with, or resided in, the infected area mentioned in my last weekly report; of the remaining 2 cases, no reliable in- formation could be obtained from one, as the patient was found migrating to a village on foot, and, when arrested by the police, was semi-delirious, the other case showing no direct signs of infection, beyond the fact that two cases had previously occurred in his neigh- bourhood. 3. The following few facts may prove interes

Cleaning folders

In [9]:
import os
import glob

#files = glob.glob(res_folder+"nls-sample/*")
#for f in files:
#    os.remove(f)